# 🧹 3장 · 지저분한 데이터 정리하기: 판다스와 데이터 정제 (개정판 실습 노트북)

『빅데이터분석 with 파이썬』 3장의 **05. 파이썬 실습**과 **미니 프로젝트**를 그대로 따라 하는 노트북입니다. 실습 번호는 책과 같습니다.

| 표시 | 뜻 | 실습 |
|---|---|---|
| ★ 필수 | 수업 시간에 모두 함께 | 3-1 ~ 3-6 |
| ☆ 선택 | 자습이나 심화 | 3-7 ~ 3-10 |

**데이터**
- 따릉이: UCI "Seoul Bike Sharing Demand" 8,760시간(2017-12-01 ~ 2018-11-30). GitHub에 올라온 사본을 바로 읽습니다.
- 미니 프로젝트: 강의 자료 **지하철_원자료_표본.zip** (서울교통공사 "역별 일별 시간대별 승하차인원 정보" 원자료 여섯 파일에서 강남·신림·홍대입구역 줄만 그대로 옮긴 것). 미니 프로젝트 전에 왼쪽 📁 파일 창에 끌어다 놓으십시오.

코드 셀은 **위에서부터 차례로** 실행하십시오(`Shift + Enter`). 아래 셀은 위에서 만든 변수를 이어서 씁니다.

## 0 · 준비

그래프의 한글이 네모(□)로 깨지지 않게 해 주는 **koreanize-matplotlib**를 설치합니다. 판다스, 넘파이, 맷플롯립은 코랩에 이미 들어 있습니다.

In [ ]:
!pip install -q koreanize-matplotlib

**데이터 받기** (깃허브 저장소판에서 덧붙인 셀) — 아래 셀은 이 장이 쓰는 강의 자료 파일을 저장소의 `data` 폴더에서 코랩으로 내려받습니다.

In [ ]:
# 이 장의 강의 자료 파일을 책 깃허브 저장소에서 받아 옵니다.
# (파일을 코랩 왼쪽 📁 파일 창에 직접 끌어다 올렸다면 이 셀은 건너뛰어도 됩니다.)
import os, urllib.parse, urllib.request
BASE = "https://raw.githubusercontent.com/yeonu69/bigdata-python-book/main/ch03/data/"
for f in [
    "bike_daily_clean.csv",
    "bike_hourly_clean.csv",
    "cleaning_log.csv",
    "지하철_3개역_정제_2021_2024.csv",
    "지하철_원자료_표본.zip",
]:
    if not os.path.exists(f):
        urllib.request.urlretrieve(BASE + urllib.parse.quote(f), f)
    print("준비됨:", f, f"({os.path.getsize(f) / 1e6:.1f} MB)")

## 05. 파이썬 실습: 정제된 하루 단위 표 만들기

이 절의 실습은 코랩(Colab)에서 위에서부터 차례로 실행합니다. 따릉이 데이터는 인터넷 주소에서 바로 읽습니다. 판다스의 함수는 대부분 원래 표를 바꾸지 않고 새 표를 돌려줍니다. 그래서 `df = …`처럼 받아 두어야 합니다. ★는 수업 시간에 꼭 돌리는 실습이고, ☆는 자습용입니다. 실행 결과는 판다스 3.0에서 돌린 것입니다. 판다스 2.x에서는 글자 열의 자료형이 `str` 대신 `object`로 보이지만 숫자는 같습니다.

#### [파이썬 실습 3-1 ★] 파일 읽기와 첫인상 살피기

따릉이 파일을 인코딩 없이 읽어 오류를 확인한 뒤 라틴-1로 다시 읽으시오. 열 이름을 짧게 바꾸고, 숫자 열의 평균, 최솟값, 중앙값, 최댓값을 살피시오.

① 한글 글꼴을 설정하고 파일을 읽습니다. ② 열 이름을 사전(dict)으로 바꿉니다. ③ `describe( )`로 요약하고 습도를 히스토그램으로 그립니다.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
try:
    import koreanize_matplotlib            # 코랩: 한글 글꼴을 자동으로 설정
except ImportError:                        # 나눔고딕 파일이 이미 있는 컴퓨터
    import glob, os
    import matplotlib.font_manager as fm
    for f in glob.glob(os.path.expanduser("~/.fonts/Nanum*.ttf")):
        fm.fontManager.addfont(f)
    plt.rcParams["font.family"] = "NanumGothic"
plt.rcParams["axes.unicode_minus"] = False

URL = ("https://raw.githubusercontent.com/syedsharin/"
       "Seoul-Bike-Sharing-Demand-Prediction/main/SeoulBikeData.csv")
try:
    pd.read_csv(URL)                        # 인코딩을 말하지 않으면 UTF-8로 읽는다
except UnicodeDecodeError as e:
    print("읽기 실패:", e)
raw = pd.read_csv(URL, encoding="latin-1")  # 원본: 이 표는 끝까지 고치지 않는다
print(raw.shape)

names = {"Date": "date", "Rented Bike Count": "count", "Hour": "hour",
         "Temperature(°C)": "temp", "Humidity(%)": "humidity",
         "Wind speed (m/s)": "wind", "Visibility (10m)": "visibility",
         "Dew point temperature(°C)": "dew", "Solar Radiation (MJ/m2)": "solar",
         "Rainfall(mm)": "rain", "Snowfall (cm)": "snow", "Seasons": "season",
         "Holiday": "holiday", "Functioning Day": "functioning"}
df = raw.rename(columns=names)               # 고친 것은 새 이름 df에 담는다
print(df[["date", "hour", "count", "temp"]].head(3))
print("date 열의 자료형:", df["date"].dtype)
print(df[["count", "temp", "humidity", "dew", "rain"]].describe().round(1).T[["mean", "min", "50%", "max"]])

df["humidity"].plot.hist(bins=50, color="#8EC5E8", edgecolor="white", figsize=(6, 3))
plt.xlabel("습도(%)")
plt.ylabel("시간 수")
plt.show()

눈여겨볼 곳은 두 군데입니다. date 열의 자료형이 `str`, 곧 날짜처럼 생긴 글자라는 점과, min 줄의 대여량 0과 습도 0입니다. `describe( )`의 min과 max는 한 줄씩 소리 내어 읽어 보십시오. "습도 0%라고? 그게 가능한가?" 같은 질문이 여기서 나옵니다.

> **[코드 읽기] `try`와 `except`**
>
> `try:` 아래의 코드를 먼저 해 봅니다. 거기서 `UnicodeDecodeError`가 나면 멈추지 않고 `except` 아래의 코드를 실행합니다. 여기서는 오류 메시지를 출력만 하고 다음 줄로 넘어갑니다. 한글 글꼴을 설정하는 맨 위의 `try`도 같은 방식입니다. 코랩에는 koreanize_matplotlib이 없으면 설치하는 셀을 먼저 실행합니다.

#### [파이썬 실습 3-2 ★] 날짜를 바르게 읽기

날짜 글자를 날짜형으로 바꾸시오. 칸마다 짐작하게 했을 때와 형식을 알려 줬을 때를 견주고, 달별 하루 평균이 어떻게 달라지는지 확인하시오.

① `format="mixed"`로 짐작하게 하고, `format="%d/%m/%Y"`로 알려 줍니다. ② 기간과 계절 열로 확인합니다. ③ 두 날짜로 달별 평균을 내어 그립니다.

In [ ]:
wrong = pd.to_datetime(df["date"], format="mixed")      # 칸마다 따로 짐작
good = pd.to_datetime(df["date"], format="%d/%m/%Y")    # 형식을 분명히 알려 줌
print("짐작한 기간:", wrong.min().date(), "~", wrong.max().date())
print("알려 준 기간:", good.min().date(), "~", good.max().date())
print("다른 날로 옮겨 간 날:", (wrong != good).sum() // 24, "일")
print(pd.crosstab(wrong.dt.month, df["season"]).loc[[1, 7]] // 24)

cmp = pd.DataFrame({"바르게": df.groupby(good.dt.month)["count"].mean() * 24,
                    "잘못": df.groupby(wrong.dt.month)["count"].mean() * 24}).round()
print(cmp.T.to_string())
print("가장 많은 달 ÷ 가장 적은 달:", (cmp.max() / cmp.min()).round(1).to_dict())
df["date"] = good                           # 이제부터는 바르게 읽은 날짜를 쓴다

cmp.plot.bar(color=["#3F7CC0", "#F2A65A"], figsize=(7, 3))
plt.xlabel("달")
plt.ylabel("하루 평균 대여량(대)")
plt.xticks(rotation=0)
plt.show()

짐작한 날짜는 2017년 1월부터 2018년 12월까지 퍼져 있고, 132일이 다른 날로 옮겨 갔습니다. `pd.crosstab( )`은 두 열의 조합마다 행 수를 세는 표입니다. 24로 나누어 날수로 바꾸었습니다. "1월"에 여름 사흘, "7월"에 겨울 사흘이 섞여 있습니다. `%d`는 일, `%m`은 월, `%Y`는 네 자리 연도입니다. 파일의 글자 모양과 같은 순서로 적어 줍니다. 달별 평균에서는 1장에서 만난 `groupby( )`를 다시 썼습니다. 같은 달의 행끼리 묶어 평균을 냅니다. 시간 평균에 24를 곱하면 하루 평균입니다.

#### [파이썬 실습 3-3 ★] 숨은 결측 찾기: 운영 중단의 0과 습도 0%

운영하지 않은 시간의 규모와 달별 분포를 확인하시오. 습도 0%인 시간을 찾아 기온과 이슬점으로 계산한 습도와 견주고, 결측으로 바꾸시오.

In [ ]:
off = df[df["functioning"] == "No"]         # 운영하지 않은 시간
print("운영 안 한 시간:", len(off), " 날짜 수:", off["date"].nunique())
print("대여량이 0인 시간:", (df["count"] == 0).sum())
print("달마다:", off.groupby(off["date"].dt.month).size().to_dict())

zero = df["humidity"] == 0
print("습도 0%:", zero.sum(), "시간", df.loc[zero, "date"].dt.strftime("%m-%d").unique().tolist())
# 기온과 이슬점으로 습도를 다시 계산해 본다 (마그누스 근사식)
calc = 100 * np.exp(17.625 * df["dew"] / (243.04 + df["dew"])) \
           / np.exp(17.625 * df["temp"] / (243.04 + df["temp"]))
print("습도가 0이 아닌 시간: 기록과 계산의 차이 평균",
      round((calc - df["humidity"])[~zero].abs().mean(), 2), "%p")
print("습도 0%인 시간의 계산값:", round(calc[zero].min()), "~", round(calc[zero].max()), "%")
w = calc[zero].idxmax()                          # 계산값이 가장 큰 시간
print("예:", df.loc[w, "date"].date(), df.loc[w, "hour"], "시, 기온", df.loc[w, "temp"], "이슬점", df.loc[w, "dew"], "→", round(calc[w]), "%")
df["humidity"] = df["humidity"].replace(0, np.nan)   # 0% → 결측(NaN)
print("습도 빈칸:", df["humidity"].isna().sum())

plt.figure(figsize=(5, 4))
plt.scatter(calc[~zero], raw.loc[~zero, "Humidity(%)"], s=3, color="#8EC5E8", label="보통 시간")
plt.scatter(calc[zero], raw.loc[zero, "Humidity(%)"], s=30, color="#E04B3C", label="습도 0%")
plt.xlabel("기온·이슬점으로 계산한 습도(%)")
plt.ylabel("기록된 습도(%)")
plt.legend()
plt.show()

`df[조건]`은 조건이 True인 행만 고르는 조건 필터입니다. 2장의 불리언 마스크와 같은 생각입니다. 대여량이 0인 시간은 정확히 운영하지 않은 295시간이고, 9~11월에 247시간이 몰려 있습니다. `replace(0, np.nan)`이 0을 NaN으로 바꾸자, 이제 `isna( )`가 이 17시간을 결측으로 알아봅니다.

#### [파이썬 실습 3-4 ★] 이상치 후보 찾기: IQR 규칙

운영한 시간의 대여량에 IQR 규칙을 쓰고, 후보가 언제인지 확인하시오. 이어서 시각마다 따로 울타리를 쳐 보시오. 강수량에도 써 보시오.

In [ ]:
h = df[df["functioning"] == "Yes"].copy()   # 운영한 시간만
q1, q3 = h["count"].quantile([0.25, 0.75])
iqr = q3 - q1
upper = q3 + 1.5 * iqr
top = h.loc[h["count"].idxmax()]
print("가장 많은 시간:", top["date"].date(), top["hour"], "시", top["count"], "대")
print(f"운영한 {len(h):,}시간  Q1 {q1:.0f}, Q3 {q3:,.0f}, IQR {iqr:.0f}, 위 울타리 {upper:,.0f}")
out = h[h["count"] > upper]
print("이상치 후보:", len(out), "시간, 많은 시각:", out["hour"].value_counts().head(2).to_dict())
print("그 가운데 평일:", round((out["date"].dt.dayofweek < 5).mean() * 100, 1), "%")

q = h.groupby("hour")["count"].quantile([0.25, 0.75]).unstack()   # 시각마다 Q1, Q3
fence = q[0.75] + 1.5 * (q[0.75] - q[0.25])                      # 시각마다 위 울타리
print(f"저녁 6시끼리: Q3 {q.loc[18, 0.75]:,.0f}, 위 울타리 {fence[18]:,.0f}")
print("시각별로 잰 이상치 후보:", (h["count"] > h["hour"].map(fence)).sum(), "시간")
print("비가 0인 시간:", round((h["rain"] == 0).mean() * 100, 1), "%,  비의 IQR:",
      h["rain"].quantile(0.75) - h["rain"].quantile(0.25), ",  비 온 시간:", (h["rain"] > 0).sum())

h.boxplot(column="count", by="hour", figsize=(8, 3.5), grid=False)
plt.axhline(upper, color="#E04B3C", linestyle="--")
plt.title("")
plt.suptitle("")
plt.xlabel("시각")
plt.ylabel("한 시간 대여량(대)")
plt.show()

`quantile([0.25, 0.75])`는 Q1과 Q3을 한 번에 돌려줍니다. `groupby("hour")`에 붙이면 시각마다 따로 구하고, `unstack( )`은 그 결과를 시각 × (Q1, Q3) 표로 폅니다. `map(fence)`는 행마다 자기 시각의 울타리를 찾아 붙입니다.

> **[코드 읽기] f-문자열의 `:,.0f`**
>
> 따옴표 앞에 f를 붙이면 중괄호 안에 변수를 넣을 수 있습니다. `{upper:,.0f}`의 `,`는 천 단위 쉼표, `.0f`는 "소수점 아래 0자리까지"라는 뜻입니다. 그래서 2389.0이 2,389로 찍힙니다.

#### [파이썬 실습 3-5 ★] 하루 단위로 묶기

운영한 시간만 날짜별로 묶어 하루 단위 표를 만들고, 24시간이 다 있는 날만 남기시오. 0을 그대로 둔 365일과 하루 평균, 9월 평균, 바쁜 달 순서를 견주시오.

① `groupby("date").agg( )`로 열마다 더할지 평균 낼지 정합니다. ② 24시간이 다 있는 날만 남깁니다. ③ 0을 둔 표와 견주어 그립니다.

In [ ]:
day = (h.groupby("date")
        .agg(count=("count", "sum"), temp=("temp", "mean"), rain=("rain", "sum"),
             snow=("snow", "sum"), humidity=("humidity", "mean"),
             season=("season", "first"), holiday=("holiday", "first"),
             hours=("hour", "size"))
        .reset_index())
print(len(day), "일 →", end=" ")
day = day[day["hours"] == 24].drop(columns="hours").reset_index(drop=True)
print(len(day), "일")

all365 = df.groupby("date")["count"].sum()               # 0을 그대로 둔 365일
print(f"하루 평균  0 포함 {all365.mean():,.0f}대,  운영한 날만 {day['count'].mean():,.0f}대")
m0 = all365.groupby(all365.index.month).mean()
m1 = day.groupby(day["date"].dt.month)["count"].mean()
print(f"9월  0 포함 {m0[9]:,.0f}대,  운영한 날만 {m1[9]:,.0f}대")
print("바쁜 달 순서  0 포함:", m0.sort_values(ascending=False).index[:4].tolist(),
      " 운영한 날만:", m1.sort_values(ascending=False).index[:4].tolist())
top, low = day.loc[day["count"].idxmax()], day.loc[day["count"].idxmin()]
print("가장 많은 날:", top["date"].date(), top["count"], " 가장 적은 날:", low["date"].date(), low["count"], "비", low["rain"], "mm")

pd.DataFrame({"0 포함": m0, "운영한 날만": m1}).plot.bar(color=["#B9C0C8", "#E04B3C"], figsize=(7, 3))
plt.xlabel("달")
plt.ylabel("하루 평균 대여량(대)")
plt.xticks(rotation=0)
plt.show()

`agg( )` 안의 `count=("count", "sum")`은 "새 열 count는 원래 count 열을 더한 것"이라는 뜻입니다. 대여량과 비는 더하고, 기온과 습도는 평균을 냈습니다. 무엇을 더하고 무엇을 평균 냈는지도 기록해 둘 결정입니다.

#### [파이썬 실습 3-6 ★] 공휴일 표와 합치고 저장하기

공식 공휴일 표를 만들어 하루 단위 표와 합치고, 원래 Holiday 열과 어긋난 날을 찾으시오. 두 기준의 공휴일 효과를 견주고, 쉬는 날 열을 만들어 저장하시오.

① `merge( )`로 날짜를 열쇠 삼아 합칩니다. ② 어긋난 날과 두 기준의 평균을 구합니다. ③ 쉬는 날 열을 만들고 CSV로 저장한 뒤 다시 읽어 확인합니다.

In [ ]:
hol = pd.DataFrame({
    "date": ["2017-12-25", "2018-01-01", "2018-02-15", "2018-02-16", "2018-02-17",
             "2018-03-01", "2018-05-05", "2018-05-07", "2018-05-22", "2018-06-06",
             "2018-06-13", "2018-08-15", "2018-09-23", "2018-09-24", "2018-09-25",
             "2018-09-26", "2018-10-03", "2018-10-09"],
    "name": ["성탄절", "신정", "설날 연휴", "설날", "설날 연휴", "삼일절", "어린이날",
             "대체공휴일", "부처님오신날", "현충일", "지방선거일", "광복절", "추석 연휴",
             "추석", "추석 연휴", "대체공휴일", "개천절", "한글날"]})
hol["date"] = pd.to_datetime(hol["date"])               # 연-월-일 글자라 헷갈릴 일이 없다
day = day.merge(hol, on="date", how="left")
day["official"] = day["name"].notna()
print(len(day), "행")
print(pd.crosstab(day["holiday"], day["official"]))
diff = day[(day["holiday"] == "Holiday") != day["official"]]
print(diff[["date", "holiday", "name", "count"]].to_string(index=False))
for label, is_hol in [("원래 열", day["holiday"] == "Holiday"), ("공식 표", day["official"])]:
    print(f"{label} 기준  공휴일 {day.loc[is_hol, 'count'].mean():,.0f}대,"
          f"  그 밖의 날 {day.loc[~is_hol, 'count'].mean():,.0f}대")

day["offday"] = (day["date"].dt.dayofweek >= 5) | day["official"]   # 주말이거나 공휴일
print(day.groupby("offday")["count"].agg(["size", "mean"]).round())
h["offday"] = (h["date"].dt.dayofweek >= 5) | h["date"].isin(hol["date"])   # 시간 표에도
print("아침 8시 평균:", h[h["hour"] == 8].groupby("offday")["count"].mean().round().to_dict())

day.to_csv("bike_daily_clean.csv", index=False, encoding="utf-8-sig")
h.to_csv("bike_hourly_clean.csv", index=False, encoding="utf-8-sig")
check = pd.read_csv("bike_daily_clean.csv", parse_dates=["date"])
print("다시 읽은 표:", check.shape, check["date"].min().date(), "~", check["date"].max().date())

h[h["hour"] == 8].groupby("offday")["count"].mean().rename({False: "평일", True: "쉬는 날"}) \
    .plot.bar(color=["#3F7CC0", "#E04B3C"], figsize=(4, 3))
plt.xlabel("")
plt.ylabel("아침 8시 평균 대여량(대)")
plt.xticks(rotation=0)
plt.show()

`how="left"`는 왼쪽 표(day)의 행을 모두 남기라는 뜻입니다. 공휴일 표에 없는 날짜는 name이 NaN이 됩니다. 합친 뒤에는 행 수(352행)부터 확인했습니다. 열쇠가 겹치면 행이 불어날 수 있기 때문입니다. `encoding="utf-8-sig"`는 이 파일을 엑셀로 열어도 한글이 깨지지 않게 해 줍니다. 저장한 파일을 다시 읽어 확인하는 것까지가 저장입니다. 4장과 5장은 이 bike_daily_clean.csv, 시간 단위의 bike_hourly_clean.csv와 같은 규칙으로 정리한 표에서 출발합니다.

#### [파이썬 실습 3-7 ☆] 결측을 채우면 무엇이 바뀌나

운영하지 않은 13일을 빈칸으로 두고, 지우기, 0으로 두기, 평균으로 채우기, 중앙값으로 채우기를 견주시오. 판다스와 넘파이의 표준편차도 견주시오.

In [ ]:
s = all365.where(~all365.index.isin(off["date"]))        # 운영하지 않은 13일 → NaN
print("빈칸:", s.isna().sum(), "일")
ways = {"지우기": s.dropna(), "0으로": s.fillna(0),
        "평균으로": s.fillna(s.mean()), "중앙값으로": s.fillna(s.median())}
print(pd.DataFrame({k: [len(v), v.mean(), v.median(), v.std()] for k, v in ways.items()},
                   index=["날수", "평균", "중앙값", "표준편차"]).round(0))
x = pd.Series([4, 6, 8, 10, 12])                       # [예제 3-3]의 다섯 날
print("판다스 std( ):", round(x.std(), 2), "  넘파이 np.std( ):", round(np.std(x), 2))

[표 3-7]의 숫자입니다. `where( )`는 조건이 True인 곳만 남기고 나머지를 NaN으로 바꿉니다. `fillna( )`는 NaN을 채우고, `dropna( )`는 NaN을 뺍니다. 마지막 줄은 [잠깐만]의 두 표준편차입니다.

#### [파이썬 실습 3-8 ☆] 정제 일지를 표로 저장하기

[표 3-5]의 정제 일지를 데이터프레임으로 만들어 CSV로 저장하시오.

In [ ]:
log = pd.DataFrame([
    ("인코딩", "latin-1로 읽음", "°C 바이트(0xB0)가 UTF-8이 아님", "14열"),
    ("날짜", "format='%d/%m/%Y'", "일/월/연 순서, 계절 열로 확인", "365일"),
    ("습도 0%", "결측(NaN)으로", "이슬점과 모순", "17시간"),
    ("운영 안 한 시간", "하루 분석에서 뺌", "0은 수요가 아니라 운영 중단", "295시간"),
    ("24시간 미만인 날", "뺌", "10월 6일은 17시간만 운영", "1일"),
    ("이상치 후보", "그대로 둠", "퇴근 시간의 진짜 값", "152시간"),
    ("공휴일", "공식 표를 합쳐 새 열로", "원래 열과 8일이 다름, 원래 열도 남김", "17일")],
    columns=["무엇을", "어떻게", "왜", "몇 건"])
log.to_csv("cleaning_log.csv", index=False, encoding="utf-8-sig")
print(log.to_string(index=False))

정제한 표와 일지를 함께 저장해 두면, 석 달 뒤의 나도 왜 10월 6일을 뺐는지 알 수 있습니다.

#### [파이썬 실습 3-9 ☆] 넓은 표와 깔끔한 표 오가기

따릉이 시간 기록을 날짜 × 시각의 넓은 표로 바꾸었다가, 다시 깔끔한 표로 되돌리시오.

In [ ]:
wide = df.pivot(index="date", columns="hour", values="count")   # 날짜 × 시각
print("넓은 표:", wide.shape)
print(wide.iloc[:2, :4])
long = wide.reset_index().melt(id_vars="date", var_name="hour", value_name="count")
print("깔끔한 표:", long.shape)
print(long.sort_values(["date", "hour"]).head(3).to_string(index=False))

`pivot( )`은 깔끔한 표를 넓은 표(365 × 24)로, `melt( )`는 넓은 표를 다시 깔끔한 표(8,760 × 3)로 바꿉니다. 미니 프로젝트의 지하철 파일이 바로 넓은 표입니다.

#### [파이썬 실습 3-10 ☆] 계산한 습도로 채우고 표시 남기기

습도 결측 17시간을 [파이썬 실습 3-3]에서 계산한 값으로 채우고, 채웠다는 표시 열을 만드시오.

In [ ]:
df["humidity_calc"] = df["humidity"].isna()             # 채웠다는 표시
df["humidity"] = df["humidity"].fillna(calc.round())
print(df.loc[df["humidity_calc"], ["date", "hour", "temp", "dew", "humidity"]].head(3).to_string(index=False))
print("채운 시간:", df["humidity_calc"].sum())

5월 19일 아침 7시는 63%로 채워졌습니다. humidity_calc 열이 True인 17시간은 잰 값이 아니라 계산한 값이라는 표시입니다.

> **핵심 3줄**
>
> 인코딩, 날짜 형식, 조건 필터, IQR 규칙, `groupby( )`, `merge( )`를 차례로 써서 정제된 하루 단위 표를 만들었습니다.
>
> 판다스 함수는 새 표를 돌려주므로 받아 두고, 단계마다 행 수와 기간을 출력해 확인합니다.
>
> 결과는 bike_daily_clean.csv(352일)와 bike_hourly_clean.csv(8,465시간)로 저장했고, 원본은 고치지 않았습니다.

## [미니 프로젝트 3] 지하철 승객은 해마다 얼마나 늘었을까

**질문.** 코로나19가 한창이던 2021년부터 2024년까지, 서울 지하철 2호선 강남역, 신림역, 홍대입구역의 승객은 얼마나 늘었을까요? 답은 공공데이터포털에 있습니다. 그런데 받은 파일을 그대로 더하면 엉뚱한 답이 나옵니다.

**데이터.** 서울교통공사가 공공데이터포털에 올린 "역별 일별 시간대별 승하차인원 정보"입니다. 한 행은 "어느 날, 어느 역의 승차(또는 하차)"이고, 열은 날짜, 호선, 역번호, 역명, 승하차 구분과 시간대별 인원입니다. 시간대가 열 이름인 넓은 표입니다. 강의 자료의 지하철_원자료_표본.zip에는 포털에서 받은 원자료 여섯 파일에서 세 역의 줄만 글자 하나 바꾸지 않고 옮겨 두었습니다(파일마다 720~2,196행). 지저분한 모습도 원자료 그대로입니다. 이 zip 파일을 코랩 왼쪽 파일 창에 끌어다 놓고 시작합니다. 정제를 마친 정답 파일(지하철_3개역_정제_2021_2024.csv)은 강의 자료에 따로 있습니다.

> **[더 알아보기] ☆ 포털에서 원자료를 직접 받기**
>
> 공공데이터포털(data.go.kr)에서 "서울교통공사 역별 일별 시간대별 승하차인원 정보"를 검색하면 해마다의 CSV 파일을 받을 수 있습니다. 한 해에 약 20만 행, 20~30MB입니다. 2023년은 4월, 9월, 12월 기준의 파일 셋을 받았는데, 기간이 서로 겹쳤습니다. 받은 파일을 subway_raw 폴더에 넣고 1단계의 zip 줄만 지우면, 같은 코드가 모든 역에 그대로 돌아갑니다. 5장의 미니 프로젝트는 이렇게 정제한 2023~2024년 자료를 서울 전체로 더해 씁니다.


**1단계: 데이터 신원 확인**([파이썬 실습 3-1] 응용). zip을 풀고 파일마다 cp949로 읽어, 행·열 수, 기간, 둘째·셋째 열 이름, 마지막 열 이름, 빈칸 수를 출력하시오.

In [ ]:
import zipfile, glob
zipfile.ZipFile("지하철_원자료_표본.zip").extractall("subway_raw")   # 강의 자료 파일
raw_s = {}
for f in sorted(glob.glob("subway_raw/*.csv")):
    t = pd.read_csv(f, encoding="cp949")       # 공공데이터포털 파일의 한글은 cp949
    tag = f[-12:-4]                            # 파일 이름 끝의 기준일, 예: 20211231
    raw_s[tag] = t
    print(tag, t.shape, t.iloc[0, 1], "~", t.iloc[-1, 1], list(t.columns[1:3]), t.columns[-1],
          "빈칸", t.isna().sum().sum())

파일마다 모양이 다릅니다. 2021년 파일은 날짜 열 이름이 "날짜"이고 맨 끝에 "합 계" 칸이 있습니다. 2022년부터는 "수송일자"이고 합계가 없습니다. 2023년은 1~4월, 1~9월, 1~12월 파일이 겹쳐 있습니다. 2022년 파일에는 빈칸이 906개 있습니다. 1~5월의 "24시이후" 칸이 모두 비어 있습니다.

**2단계: 기준선 만들기.** 정제하지 않고, 파일마다 숫자 열을 모두 더해 해마다 하루 평균 승객을 구하시오. 연번과 역번호 열만 빼시오.

In [ ]:
naive = {}
for tag, t in raw_s.items():                   # 정제하지 않고: 숫자 열을 모두 더한다
    num = t.select_dtypes("number").drop(columns=["연번", "역번호"], errors="ignore")
    naive[tag[:4]] = naive.get(tag[:4], 0) + num.sum().sum()
days = {"2021": 365, "2022": 365, "2023": 365, "2024": 366}
naive = pd.Series({y: naive[y] / days[y] for y in days}).round()
print("정제 전 하루 평균(세 역):", naive.astype(int).to_dict())

정제 전 숫자는 2021년 66만 명에서 2022년 37만 명으로 반 토막이 났다가, 2023년 81만 명으로 두 배가 됩니다. 말이 되지 않습니다. 이 들쭉날쭉한 숫자가 정제로 고쳐야 할 출발점입니다.

**3단계: 열 맞추고 잇고 겹침 지우기**([파이썬 실습 3-6], [3-9] 응용). 열 이름과 시간대 칸을 해마다 같게 맞추어 잇고, 같은 날짜·역·구분이 겹치면 가장 나중 파일을 남기시오.

In [ ]:
hours = ["06시이전"] + [f"{k:02d}-{k + 1:02d}시간대" for k in range(6, 23)] + ["23시이후"]
parts = []
for tag, t in raw_s.items():
    t = t.copy()
    if "합 계" in t.columns:                       # 2021년: 시간대 19칸 + 합계
        t = t.drop(columns="합 계")
    else:                                          # 2022년부터: 20칸 → 23시 이후를 한 칸으로
        t["23시이후"] = t["23-24시간대"] + t["24시이후"].fillna(0)   # 빈칸은 0으로 본다
        t = t.drop(columns=["23-24시간대", "24시이후"])
    t.columns = ["연번", "날짜", "호선", "역번호", "역명", "구분"] + hours
    t["호선"] = t["호선"].astype(str).str.replace("호선", "") + "호선"   # 2 → 2호선
    t["파일"] = tag
    parts.append(t)
sub = pd.concat(parts, ignore_index=True)
sub["날짜"] = pd.to_datetime(sub["날짜"])
print("이어 붙인 행:", len(sub))
key = ["날짜", "역명", "구분"]
print("같은 날짜·역·구분이 겹친 행:", sub.duplicated(key, keep=False).sum())
q3_23 = sub[(sub["날짜"] >= "2023-07-01") & (sub["날짜"] <= "2023-09-30")]
print("2023년 7~9월 합계:", q3_23.groupby("파일")[hours].sum().sum(axis=1).to_dict())
sub = sub.sort_values(["파일"] + key).drop_duplicates(key, keep="last")   # 가장 나중 파일을 믿는다
print("겹침을 지운 뒤:", len(sub), "행,", sub["날짜"].nunique(), "일")

2021년의 "23시 이후"는 한 칸이고, 2022년부터는 "23-24시간대"와 "24시이후" 두 칸입니다. 그래서 2022년부터의 두 칸을 더해 한 칸으로 맞추었습니다. 2022년 1~5월의 빈칸은 0으로 보았습니다. 같은 날짜·역·구분이 겹친 행은 3,996행이었습니다. 겹친 행의 값이 늘 같지는 않았습니다. 2023년 7~9월 합계가 9월 파일은 34,413,858명, 12월 파일은 35,351,557명이었습니다. 나중 파일에서 값이 고쳐진 것으로 보고, 가장 나중 파일을 믿었습니다. 겹침을 지우자 1,461일 × 6행 = 8,766행이 남았습니다.

**4단계: 이상한 값 찾고 결과 견주기**([파이썬 실습 3-4], [3-10] 응용). 달마다 "13~14시 ÷ 12~13시"를 구해 튀는 달을 찾으시오. 그 칸을 앞뒤 시간대의 평균으로 채우고 표시 열을 남긴 뒤, 해마다 하루 평균 승객을 정제 전과 견주시오.

In [ ]:
sub["달"] = sub["날짜"].dt.strftime("%Y-%m")
ratio = (sub.groupby("달")["13-14시간대"].sum() / sub.groupby("달")["12-13시간대"].sum()).round(2)
print("13~14시 ÷ 12~13시가 가장 큰 달:", ratio.sort_values().tail(3).to_dict())
bad = sub["달"] == "2024-04"
sub["채움"] = bad                                   # 고친 칸이 있는 행 표시
sub.loc[bad, "13-14시간대"] = ((sub.loc[bad, "12-13시간대"] + sub.loc[bad, "14-15시간대"]) / 2).round()
sub["합계"] = sub[hours].sum(axis=1)
clean = sub.groupby(sub["날짜"].dt.year)["합계"].sum() / sub.groupby(sub["날짜"].dt.year)["날짜"].nunique()
clean.index = clean.index.astype(str)
result = pd.DataFrame({"정제 전": naive, "정제 후": clean.round()})
print(result.astype(int).to_string())
cols = ["날짜", "호선", "역번호", "역명", "구분"] + hours + ["합계", "채움", "파일"]
sub[cols].to_csv("지하철_3개역_정제_2021_2024.csv", index=False, encoding="utf-8-sig")

(result / 10000).plot.bar(color=["#B9C0C8", "#E04B3C"], figsize=(6, 3))
plt.ylabel("하루 평균 승하차(만 명)")
plt.xlabel("")
plt.xticks(rotation=0)
plt.show()

보통 13~14시 승객은 12~13시와 비슷합니다(1.0~1.1배). 그런데 2024년 4월만 7.59배였습니다. 신림역 승차는 4월 13~14시 평균이 33,144명으로, 다른 달(약 2,600명)의 열두 배가 넘습니다. 이 값은 포털에 올라온 원자료에 그대로 있습니다. 데이터 과학자 스키나는 기록이 잘못된 오류(error)와, 자료를 만들고 옮기는 과정이 남긴 흔적(artifact)을 구별하라고 합니다. 이 칸은 오류로 보고 채웠습니다. 5장의 미니 프로젝트는 같은 오류가 있는 2024년 4월을 통째로 빼기로 합니다. 어느 쪽이든 일지에 남기면 됩니다. 2022년 1~5월의 빈칸은 흔적일 수도 있습니다. 그때 자정 넘어 다닌 열차가 없었는지, 집계가 빠졌는지는 파일만으로 알 수 없습니다.

정제한 뒤 세 역의 하루 평균 승객은 2021년 329,944명에서 2024년 406,172명으로 해마다 꾸준히 늘었습니다. 정제 전의 "반 토막, 두 배"는 합계 칸을 한 번 더 더하고(2021년), 겹친 파일을 모두 더한(2023년) 탓이었습니다.


**5단계: 한계 쓰기.** 이 정제가 아직 모르는 것을 [표 3-6]처럼 세 가지 이상 쓰시오. 2024년 4월 13~14시를 채운 방법과, 2022년 빈칸을 0으로 본 결정은 반드시 다루어야 합니다.

**제출물.** 그림 2장(정제 전후의 해마다 하루 평균, 13~14시 ÷ 12~13시의 달별 변화), 정제 일지 1개([표 3-5] 형식), 결론 5문장 이내.

**[표 3-8] 미니 프로젝트 채점 기준**

| 항목(비중) | 상 | 중 | 하 |
|---|---|---|---|
| 데이터 점검(20%) | 인코딩, 열 이름, 겹친 기간, 빈칸을 파일마다 밝힘 | 일부만 확인함 | 점검 없이 바로 더함 |
| 기준선 비교(20%) | 정제 전 숫자를 구하고 무엇이 그 숫자를 비틀었는지 설명함 | 정제 전 숫자만 구함 | 정제 전후를 견주지 않음 |
| 기법 적용(30%) | 열 맞추기, 겹침 지우기(나중 파일), 이상한 칸 처리와 표시 열이 모두 맞음 | 하나가 빠지거나 틀림 | 코드가 돌지 않거나 결과가 틀림 |
| 해석과 한계(30%) | 결정마다 "왜"를 일지에 쓰고, 한계를 셋 이상 씀 | 한계를 한두 가지 씀 | 숫자만 나열함 |

**교수용 정답 예시.** 정제 후 세 역의 하루 평균 승하차는 2021년 329,944명, 2022년 368,691명, 2023년 392,876명, 2024년 406,172명으로, 2021년보다 2024년이 23.1% 많습니다. 정제 전 숫자는 659,901명, 370,085명, 812,919명, 415,888명입니다. 2021년은 "합 계" 칸이 시간대 칸과 함께 더해져 두 배가 되었고, 2023년은 겹친 파일 세 개가 모두 더해졌습니다. 2024년의 차이는 4월 13~14시 오류 때문입니다. 겹친 3,996행 가운데 2023년 7~9월 값은 9월 파일과 12월 파일이 달랐고(12월 파일이 2.7% 많음), 학생이 9월 파일을 남겼다면 2023년 숫자가 조금 작게 나옵니다. 전체 풀이는 교수용 노트북에 있습니다.

## ✏️ [연습문제 3-6] 아침 8시끼리의 울타리

[파이썬 실습 3-4]의 `h`에서 아침 8시만 골라 Q1, Q3, 위 울타리를 구하시오. 전체에 쓴 울타리(2,389대)와 견주고, 전체 기준으로 후보였던 8시가 8시 기준으로도 후보인지 확인하시오.

In [ ]:
# 여기에 직접 코드를 써 보십시오.

### 정답 코드

직접 해 본 뒤에 실행해 보십시오.

In [ ]:
eight = h[h["hour"] == 8]["count"]                    # 아침 8시만
a, b = eight.quantile([0.25, 0.75])
print(f"아침 8시 {len(eight)}시간: Q1 {a:.0f}, Q3 {b:,.0f}, 위 울타리 {b + 1.5 * (b - a):,.1f}")
print("전체 기준으로 후보였던 8시:", (eight > upper).sum(), "시간,  8시 기준 후보:", (eight > b + 1.5 * (b - a)).sum(), "시간")